# SARA large (~551M) — Kaggle training on public + your datasets
**GPU requirement:** P100 (16GB, bf16 via fp16 autocast) ya T4 x2 with small batch.
**Honest note:** full pretraining of 550M on free Kaggle is ~1-2 weeks of sessions.
**Recommended strategy:** train `medium` (132M) first — it fits free GPUs comfortably. Use `large` when you have IndiaAI/rented A100s.

| Cell | Kaam |
|---|---|
| 1 | Clone + install |
| 2 | Data: public HF dataset (openwebtext sample / tinystories) |
| 3 | Corpus build |
| 4 | Training (resume-able) |
| 5 | Generate |
| 6 | Save output |

In [ ]:
# 1. Clone + install
!git clone https://github.com/skmandal3240/SARA /kaggle/working/SARA
%cd /kaggle/working/SARA
!pip install -q -r requirements.txt pyyaml pymupdf gdown datasets

In [ ]:
# 2. Build big corpus: public dataset + your Drive PDFs (optional)
import os
from datasets import load_dataset

# Public English corpus (~2GB text). Change as needed:
ds = load_dataset('roneneldan/TinyStories', split='train', streaming=True)
parts = []
total = 0
TARGET_CHARS = 2_000_000_000 // 4   # ~500M chars cap for disk/time; raise if you like
for i, ex in enumerate(ds):
    t = ex['text'].strip()
    if len(t) >= 200:
        parts.append(t)
        total += len(t)
    if total > TARGET_CHARS or i > 800000:
        break
print(f'{len(parts)} docs, {total/1e6:.0f}M chars from TinyStories')

# append your own PDFs corpus if attached (optional)
if os.path.exists('data/drive_corpus.txt'):
    parts.append(open('data/drive_corpus.txt', encoding='utf-8').read())
    print('drive corpus merged')

os.makedirs('data', exist_ok=True)
with open('data/public_corpus.txt', 'w', encoding='utf-8') as f:
    f.write('\n\n<|doc|>\n\n'.join(parts))
print('corpus MB:', os.path.getsize('data/public_corpus.txt')//1048576)

In [ ]:
# 3. Tokenize (32k vocab) + build train.bin
!python prepare_data.py --config configs/sara_large.yaml --vocab-size 32768 --source file:data/public_corpus.txt

In [ ]:
# 4. TRAIN large — resume-able across sessions (--init-from)
# Session 1: !python train_large.py --steps 60000 --batch 4 --accum 4 --seq 1024
# Session 2+: attach previous checkpoint as Kaggle Dataset, then:
!python train_large.py --steps 60000 --batch 4 --accum 4 --seq 1024 --init-from checkpoints/sara_large/sara.pt 2>/dev/null || python train_large.py --steps 60000 --batch 4 --accum 4 --seq 1024

In [ ]:
# 5. Generate
import torch
from pathlib import Path
from generate import load_sara
model, tok, cfg = load_sara(Path('checkpoints/sara_large/sara.pt'))
ids = torch.tensor([tok.encode(tok.wrap_user('Once upon a time'))], dtype=torch.long)
out = model.generate(ids, max_new=120, temperature=0.8, eos_id=tok.eos_id)
print(tok.decode(out[0].tolist()))

In [ ]:
# 6. Save output (~2.2 GB fp32 checkpoint)
import shutil
shutil.copy('checkpoints/sara_large/sara.pt', '/kaggle/working/sara_large_trained.pt')
print('Output tab -> download')